# 🦷 Session 2 Advanced Lab — 3D CBCT for Dental AI

## From 2D images to 3D volumes

This original workshop lab introduces **3D dental imaging AI** using a lightweight, workshop-safe pipeline:

**Explore a 3D volume → understand voxel spacing → annotate in 3D → split by patient → train a tiny 3D U-Net → evaluate Dice → inspect predictions**

The default exercise generates small **synthetic CBCT-like volumes** so everyone can complete the lab in Colab without downloading patient data or a very large dataset.

> **Educational demonstration only.** The synthetic volumes are not diagnostic images and the model is not clinically validated.

### Why this lab?
A panoramic radiograph is a 2D image. A CBCT is a **3D volume**. That changes how we store, annotate, preprocess, train, evaluate, and split the data.

## 🎯 Learning objectives

By the end of this lab, you should be able to:

1. Explain the difference between a 2D image and a 3D CBCT volume.
2. Read volume **shape, voxel spacing, and orientation**.
3. View axial, coronal, and sagittal planes.
4. Explain how 3D segmentation masks are created in **3D Slicer**.
5. Keep the image and segmentation mask spatially aligned.
6. Explain why **patients — not slices — must be split** into train/validation/test.
7. Train a small **3D U-Net** demonstration with MONAI.
8. Interpret the **Dice score** and inspect segmentation errors.
9. Explain what would be required before using the same workflow on real dental CBCT data.

## Part 1 — What changes when we move from 2D to 3D?

A 2D image is approximately:

`height × width`

A CBCT volume is approximately:

`depth × height × width`

Each 3D pixel is called a **voxel**.

The physical size of a voxel is described by **voxel spacing**, for example:

`0.3 mm × 0.3 mm × 0.3 mm`

A segmentation mask should match the image geometry:

- same dimensions
- same spacing
- same orientation
- same spatial origin / affine relationship

### Key lesson

**A CBCT is not just a folder of unrelated 2D slices.**

## 🟦 RUN THIS — Install the lightweight 3D imaging stack

In [ ]:
!pip -q install "monai>=1.4" nibabel plotly scikit-image

## 🟦 RUN THIS — Imports and device check

In [ ]:
from pathlib import Path
import random, math, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib

import torch
from torch.utils.data import DataLoader
from monai.data import Dataset, decollate_batch
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, EnsureTyped,
    RandCropByLabelClassesd, RandFlipd, RandRotate90d
)
from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.transforms import AsDiscrete

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Part 2 — Generate a workshop-safe synthetic 3D dental dataset

The next cell creates small **CBCT-like teaching volumes**.

These are intentionally simplified. They contain:
- background
- a jaw-like bony structure
- several tooth-like structures

The label map is:

`0 = background`  
`1 = jaw-like bone`  
`2 = tooth-like structures`

This lets us learn the mechanics of 3D AI without sharing patient data.

In [ ]:
DATA_ROOT = Path("/content/dental_cbct_3d_lab")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

SHAPE = (96, 96, 96)  # z, y, x
SPACING_MM = (0.4, 0.4, 0.4)

def ellipsoid_mask(z, y, x, cz, cy, cx, rz, ry, rx):
    return (((z-cz)/rz)**2 + ((y-cy)/ry)**2 + ((x-cx)/rx)**2) <= 1

def make_synthetic_case(case_id, seed):
    rng = np.random.default_rng(seed)
    z, y, x = np.indices(SHAPE)

    # Normalized synthetic intensity volume; NOT a real CBCT HU model
    image = rng.normal(-0.85, 0.05, SHAPE).astype(np.float32)
    label = np.zeros(SHAPE, dtype=np.uint8)

    shift_x = rng.integers(-3, 4)
    shift_y = rng.integers(-2, 3)

    jaw = ellipsoid_mask(z, y, x, 52, 58+shift_y, 48+shift_x, 18, 24, 38)
    jaw &= (y > 47 + shift_y)
    label[jaw] = 1
    image[jaw] = rng.normal(0.35, 0.08, jaw.sum())

    tooth_xs = [27, 35, 43, 53, 61, 69]
    for i, tx in enumerate(tooth_xs):
        ty = 42 + int(4 * abs((tx-48)/22))
        tz = 45 + rng.integers(-2, 3)
        tooth = ellipsoid_mask(
            z, y, x,
            tz, ty+shift_y, tx+shift_x,
            10+rng.integers(-1,2), 5, 4
        )
        label[tooth] = 2
        image[tooth] = rng.normal(0.85, 0.05, tooth.sum())

    # small acquisition-like noise/gradient for teaching
    image += ((z.astype(np.float32) / SHAPE[0]) - 0.5) * 0.05
    image = np.clip(image, -1.0, 1.0)

    affine = np.diag([SPACING_MM[2], SPACING_MM[1], SPACING_MM[0], 1.0])

    image_path = DATA_ROOT / f"{case_id}_image.nii.gz"
    label_path = DATA_ROOT / f"{case_id}_label.nii.gz"

    # nibabel uses x,y,z array convention, so transpose from z,y,x
    nib.save(nib.Nifti1Image(np.transpose(image, (2,1,0)), affine), image_path)
    nib.save(nib.Nifti1Image(np.transpose(label, (2,1,0)), affine), label_path)

    return str(image_path), str(label_path)

cases = []
for i in range(6):
    image_path, label_path = make_synthetic_case(f"case_{i+1:02d}", SEED+i)
    cases.append({"patient": f"case_{i+1:02d}", "image": image_path, "label": label_path})

pd.DataFrame(cases)

## 🟩 DISCUSS — Why synthetic data here?

This synthetic dataset is used only to make the mechanics of 3D segmentation easy to run during a short workshop.

It **does not** replace independent real dental CBCT data for:
- clinical validation
- publication claims
- generalizability testing
- diagnostic use

For real research, use a properly governed dental CBCT dataset with documented acquisition, labels, patient-level identifiers, licensing, and an independent validation plan.

## 🟦 RUN THIS — Inspect one 3D volume and its metadata

In [ ]:
sample_img = nib.load(cases[0]["image"])
sample_lab = nib.load(cases[0]["label"])

img_xyz = sample_img.get_fdata()
lab_xyz = sample_lab.get_fdata().astype(np.uint8)

print("Image shape:", img_xyz.shape)
print("Label shape:", lab_xyz.shape)
print("Voxel spacing (mm):", sample_img.header.get_zooms()[:3])
print("Affine:\n", sample_img.affine)
print("Unique labels:", np.unique(lab_xyz))

assert img_xyz.shape == lab_xyz.shape, "Image and label shapes do not match"
assert np.allclose(sample_img.affine, sample_lab.affine), "Image and label geometry do not match"
print("✓ Image and label geometry match.")

## 🟦 RUN THIS — View axial, coronal, and sagittal planes

A 3D volume should be inspected in all three orthogonal planes.

In [ ]:
cx, cy, cz = [s//2 for s in img_xyz.shape]

fig, axes = plt.subplots(2, 3, figsize=(13, 8))

axes[0,0].imshow(img_xyz[:,:,cz].T, cmap="gray", origin="lower")
axes[0,0].set_title("Axial")
axes[0,1].imshow(img_xyz[:,cy,:].T, cmap="gray", origin="lower")
axes[0,1].set_title("Coronal")
axes[0,2].imshow(img_xyz[cx,:,:].T, cmap="gray", origin="lower")
axes[0,2].set_title("Sagittal")

axes[1,0].imshow(lab_xyz[:,:,cz].T, cmap="viridis", origin="lower", vmin=0, vmax=2)
axes[1,0].set_title("Axial label")
axes[1,1].imshow(lab_xyz[:,cy,:].T, cmap="viridis", origin="lower", vmin=0, vmax=2)
axes[1,1].set_title("Coronal label")
axes[1,2].imshow(lab_xyz[cx,:,:].T, cmap="viridis", origin="lower", vmin=0, vmax=2)
axes[1,2].set_title("Sagittal label")

for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 🟨 YOUR TURN — 3D data questions

Before moving on, answer:

1. Why is voxel spacing clinically important?
2. What could happen if the image is resampled but the mask is not?
3. Why should we inspect axial, coronal, and sagittal views?
4. Is a 96×96×96 volume equal to 96 independent patients? Why not?

## Part 3 — How to annotate a real dental CBCT in 3D Slicer

For a real workshop CBCT, use **3D Slicer** rather than drawing masks slice-by-slice in a generic 2D annotation tool.

### Recommended workflow

1. Import the CBCT DICOM series into **DICOM** in 3D Slicer.
2. Open **Segment Editor**.
3. Create clearly named segments, for example:
   - Tooth
   - Jawbone
   - Mandibular canal
   - Lesion
4. Use tools such as:
   - Paint / Draw
   - Threshold
   - Grow from Seeds
   - Scissors
   - Fill Between Slices
5. Inspect the annotation in **axial, coronal, sagittal, and 3D views**.
6. Correct boundaries manually where needed.
7. Have the annotation reviewed by an appropriately qualified dental expert.
8. Export:
   - image volume
   - matching segmentation
   - ideally as NIfTI (.nii.gz) or NRRD for AI development

### Critical rule

**The segmentation mask must remain spatially aligned with the CBCT volume.**

Do not independently resize, rotate, crop, or reorient the image and label without applying the same transform to both.

## Optional — AI-assisted annotation with MONAI Label

After demonstrating manual annotation, **MONAI Label** can be introduced as an advanced AI-assisted workflow:

**CBCT → initial AI suggestion → expert correction → improved label**

The expert remains responsible for checking the annotation.

For a short Session 2 workshop, demonstrate this concept briefly rather than trying to configure a full MONAI Label server during class.

## Part 4 — Split the DATA by patient, not by slice

This is one of the most important rules in medical imaging AI.

### ❌ Wrong

Patient A:
- slices 1–100 → train
- slices 101–150 → test

The model has already seen the same patient's anatomy.

### ✅ Better

- Patient A → train
- Patient B → validation
- Patient C → test

The synthetic cases below each represent one independent teaching volume.

In [ ]:
train_cases = cases[:4]
val_cases   = cases[4:5]
test_cases  = cases[5:6]

print("TRAIN:", [x["patient"] for x in train_cases])
print("VAL:  ", [x["patient"] for x in val_cases])
print("TEST: ", [x["patient"] for x in test_cases])

assert set(x["patient"] for x in train_cases).isdisjoint(x["patient"] for x in test_cases)
print("✓ No patient/volume appears in both train and test.")

## Part 5 — Prepare 3D patches for training

Full dental CBCT volumes can be large. A common strategy is to train on **3D patches** rather than loading the entire full-resolution volume through the network at once.

In this demonstration we use 64×64×64 voxel patches.

The transform also applies simple 3D augmentation to **training data only**.

In [ ]:
train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    RandCropByLabelClassesd(
        keys=["image", "label"],
        label_key="label",
        spatial_size=(64, 64, 64),
        ratios=[1, 1, 1],
        num_classes=3,
        num_samples=2,
    ),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    RandRotate90d(keys=["image", "label"], prob=0.3, max_k=3),
    EnsureTyped(keys=["image", "label"]),
])

plain_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    EnsureTyped(keys=["image", "label"]),
])

train_ds = Dataset(train_cases, transform=train_transforms)
val_ds   = Dataset(val_cases, transform=plain_transforms)
test_ds  = Dataset(test_cases, transform=plain_transforms)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)

batch = next(iter(train_loader))
print("Training image patch shape:", tuple(batch["image"].shape))
print("Training label patch shape:", tuple(batch["label"].shape))

## 🟩 DISCUSS — Preprocessing real CBCT data

Real CBCT preprocessing must be justified and consistent.

Common considerations include:
- orientation standardization
- voxel-spacing resampling
- cropping / field-of-view handling
- intensity normalization
- artifact handling
- image/label alignment

### Important CBCT caution

CBCT intensity values are **not universally standardized across scanners in the same way as conventional medical CT Hounsfield Units**.

Do not apply a CT windowing rule blindly to every CBCT dataset. Inspect how the specific dataset was acquired and stored.

## Part 6 — Build a tiny 3D U-Net

A 3D U-Net processes **volumetric patches** and predicts a class for each voxel.

For workshop speed, this network is deliberately small.

In [ ]:
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=3,
    channels=(8, 16, 32, 64),
    strides=(2, 2, 2),
    num_res_units=1,
).to(DEVICE)

loss_function = DiceCELoss(to_onehot_y=True, softmax=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 🟨 YOUR TURN — Predict before training

What do you expect?

- At epoch 0, should the model already segment the synthetic jaw and teeth well?
- Which is harder: classifying one whole image or assigning a class to every voxel?
- Why might a small anatomical structure be more difficult to segment than a large one?

## 🟦 RUN THIS — Train for a few workshop epochs

This is intentionally short. The goal is to understand the workflow, not to optimize a clinical model.

In [ ]:
EPOCHS = 3  # keep small for the workshop

for epoch in range(EPOCHS):
    model.train()
    losses = []

    for batch_data in train_loader:
        images = batch_data["image"].to(DEVICE)
        labels = batch_data["label"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    print(f"Epoch {epoch+1}/{EPOCHS} — mean training loss: {np.mean(losses):.4f}")

## Part 7 — Validate with sliding-window inference

A full 3D volume may be too large to process at once.

**Sliding-window inference** predicts overlapping 3D patches and combines them to reconstruct the full-volume segmentation.

In [ ]:
post_pred = AsDiscrete(argmax=True, to_onehot=3)
post_label = AsDiscrete(to_onehot=3)
dice_metric = DiceMetric(include_background=False, reduction="mean")

model.eval()
with torch.no_grad():
    for val_data in val_loader:
        val_images = val_data["image"].to(DEVICE)
        val_labels = val_data["label"].to(DEVICE)

        val_outputs = sliding_window_inference(
            val_images,
            roi_size=(64, 64, 64),
            sw_batch_size=2,
            predictor=model,
            overlap=0.25,
        )

        preds = [post_pred(x) for x in decollate_batch(val_outputs)]
        labs  = [post_label(x) for x in decollate_batch(val_labels)]
        dice_metric(y_pred=preds, y=labs)

    val_dice = dice_metric.aggregate().item()
    dice_metric.reset()

print(f"Validation mean Dice (jaw + teeth): {val_dice:.3f}")

## What is Dice?

Dice measures overlap between the predicted segmentation and the reference mask.

`Dice = 2 × overlap / (prediction + reference)`

Conceptually:
- 1.0 = perfect overlap
- 0.0 = no overlap

But a high Dice score alone does **not** establish clinical safety.

Always inspect:
- which structures fail
- where boundaries fail
- whether errors are clinically important
- performance on independent patients and external data

## 🟦 RUN THIS — Evaluate the held-out synthetic test volume

In [ ]:
model.eval()
test_data = next(iter(test_loader))
test_image = test_data["image"].to(DEVICE)
test_label = test_data["label"].to(DEVICE)

with torch.no_grad():
    test_logits = sliding_window_inference(
        test_image,
        roi_size=(64, 64, 64),
        sw_batch_size=2,
        predictor=model,
        overlap=0.25,
    )

test_pred = torch.argmax(test_logits, dim=1, keepdim=True)

dice_metric.reset()
dice_metric(
    y_pred=[post_pred(x) for x in decollate_batch(test_logits)],
    y=[post_label(x) for x in decollate_batch(test_label)]
)
test_dice = dice_metric.aggregate().item()
dice_metric.reset()

print(f"Test mean Dice (synthetic held-out volume): {test_dice:.3f}")

## 🟦 RUN THIS — Inspect the prediction visually

Metrics should always be paired with visual error inspection.

In [ ]:
img = test_image[0,0].detach().cpu().numpy()
ref = test_label[0,0].detach().cpu().numpy()
pred = test_pred[0,0].detach().cpu().numpy()

z = img.shape[0] // 2

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(img[z], cmap="gray")
axes[0].set_title("Synthetic CBCT-like slice")
axes[1].imshow(ref[z], cmap="viridis", vmin=0, vmax=2)
axes[1].set_title("Reference mask")
axes[2].imshow(pred[z], cmap="viridis", vmin=0, vmax=2)
axes[2].set_title("3D U-Net prediction")
axes[3].imshow(img[z], cmap="gray")
axes[3].imshow(pred[z], cmap="viridis", alpha=0.4, vmin=0, vmax=2)
axes[3].set_title("Prediction overlay")

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 🟨 YOUR TURN — Error analysis

Look at the overlay and discuss:

1. Does the model miss any tooth-like structures?
2. Does it confuse jaw-like bone and tooth-like structures?
3. Are boundary errors visible?
4. Would the same error matter equally for:
   - tooth segmentation
   - mandibular canal segmentation
   - lesion segmentation?
5. What external data would you need before trusting a real CBCT model?

## Part 8 — Optional real CBCT exploration

For real research, use a properly licensed dataset or institutionally approved de-identified data.

A useful public research resource is the **ToothFairy2** CBCT segmentation dataset, which provides maxillofacial CBCT volumes and multi-structure annotations. Its official training data require registration/download from the project website.

This notebook does **not** redistribute ToothFairy2 data and does not copy another workshop's notebooks.

Official dataset page:
https://toothfairy2.grand-challenge.org/dataset/

### If you have one de-identified NIfTI image + matching label

Place them in Colab and use the next optional cell only to **inspect geometry and planes** before adapting the training pipeline.

In [ ]:
# OPTIONAL REAL-DATA INSPECTION — set paths only if you have approved/de-identified files
REAL_IMAGE_PATH = None   # e.g. "/content/my_cbct.nii.gz"
REAL_LABEL_PATH = None   # e.g. "/content/my_mask.nii.gz"

if REAL_IMAGE_PATH and REAL_LABEL_PATH:
    real_img = nib.load(REAL_IMAGE_PATH)
    real_lab = nib.load(REAL_LABEL_PATH)

    print("Image shape:", real_img.shape)
    print("Label shape:", real_lab.shape)
    print("Image spacing:", real_img.header.get_zooms()[:3])
    print("Label spacing:", real_lab.header.get_zooms()[:3])
    print("Unique labels:", np.unique(real_lab.get_fdata()).astype(int))

    assert real_img.shape == real_lab.shape, "Image/label shape mismatch"
    assert np.allclose(real_img.affine, real_lab.affine), "Image/label affine mismatch"
    print("✓ Basic image/label geometry check passed.")
else:
    print("Optional real-data inspection skipped.")

## Part 9 — 2D vs 3D: choose the right workflow

| Problem | Data | Annotation | Example model |
|---|---|---|---|
| Implant brand from one cropped radiograph | 2D image | Image label | CNN / YOLO classification |
| Locate implants in panoramic image | 2D image | Bounding boxes | YOLO detection |
| Outline lesion in one image | 2D image | 2D mask | U-Net / segmentation model |
| Segment teeth/jaw/canal in CBCT | 3D volume | 3D voxel mask | 3D U-Net / nnU-Net / MONAI workflow |

### Core message

**The AI task determines the annotation and model — not simply the fact that the data came from dentistry.**

## Part 10 — Workshop validation checklist

Before calling a real 3D dental model “validated,” ask:

- Are train/validation/test splits patient-independent?
- Are repeated scans from one patient kept in one split?
- Are annotations clinically defined and expert-reviewed?
- Are image and mask geometry preserved?
- Are scanner/site differences represented?
- Are small structures evaluated separately?
- Are Dice/IoU paired with clinically meaningful error analysis?
- Has the model been tested on an independent external dataset?
- Are inference failures and out-of-distribution cases handled?
- Is the intended use clearly defined?

> **High Dice on one internal dataset ≠ clinical validation.**

## ✅ Session 2 — 3D CBCT checkpoint

You should now be able to explain:

**DICOM / NIfTI → Volume → Voxel spacing → 3D annotation → Patient-level split → 3D patches → 3D U-Net → Dice → Visual QC → External validation**

### Suggested classroom flow

1. Show a CBCT in **3D Slicer**.
2. Create one small manual segment.
3. Run this Colab notebook.
4. Compare 2D implant classification with 3D voxel segmentation.
5. Finish with the validation checklist.

### Take-home message

**2D computer vision asks what is in an image or where it is.  
3D dental AI must also preserve anatomy, geometry, and patient-level independence across an entire volume.**